# How to setup vs code with databricks
## Step 1 : Install databricks extension on your vs code
## Step 2 : Create a databricks free account and create a workspace in your account.
Your workspace url should look like this : https://dbc-e59b7df7-56be.cloud.databricks.com/editor/folders/home?o=1238258517794919
## Step 3 : Paste this workspace url in your vs code 
![workspace_url](/working_with_databricks_and_pyspark/images/workspace_url.png)
You can activate this above option by clicking on this button as shown below
![activate_workspace_url](/working_with_databricks_and_pyspark/images/activate_workspace_url.png)
## Step 4 : Create a python3.11 env using conda env manager
Use this command to create a new env using conda : ```conda create -n databricks python=3.11```
## Step 5 : activate your databricks env and install databricks-connect
Activate your databricks env
```conda activate databricks```

Install databricks-connect
```pip install databricks-connect```
## Step 6 : Setup your databricks creds in the zsh env conf file
Set your databricks workspace url and your auth token 
```bash
export DATABRICKS_HOST="https://<your-workspace>.cloud.databricks.com"
export DATABRICKS_TOKEN="<paste-your-token>"
export DATABRICKS_CLUSTER_ID="0204-153214-vvzq2sph"
```
Reload your zsh shell after setting up the workspace urls and your auth tokens
```source ~/.zshrc ```

In order to get the url of your databricks workspace you need to copy the url from your browser after opening the workspace in databricks in your browser
As far as auth token is concerned then you need to generate it from user > settings > developer > token then there you need to generate your auth token
You can get you databricks cluster id by following the steps below : 
- Open Databricks workspace
- Go to Compute
- Click your cluster
- Look at the URL in your browser:
    - THE URL IN YOUR BROWSER WILL LOOK SOMETHING LIKE THIS  : ```https://dbc-xyz.cloud.databricks.com#cluster/<THIS_IS_CLUSTER_ID>```
## Step 7 : Install some databricks utilities
```bash
pip install databricks-sdk
pip install databricks-cli
```



# Databricks data engineering :
## Data ingestion : 
```bash
Recommended project structure : 
your_project/
│
├── lakeflow/
│   ├── connections/
│   │   └── postgres_conn.yaml
│   ├── ingest/
│   │   └── orders_ingest.yaml
│   └── pipelines/
│       └── README.md
│
├── src/
│   └── your_app/
│       ├── __init__.py
│       ├── bronze_to_silver/
│       │   ├── __init__.py
│       │   └── process_orders.py
│       ├── utils/
│       │   ├── __init__.py
│       │   └── spark_init.py
│       └── config/
│           └── settings.py
│
├── notebooks/
│   ├── exploration/
│   └── debugging/
│
├── tests/
│   ├── test_process_orders.py
│   └── test_utils.py
│
├── jobs/
│   └── process_orders_job.yaml
│
├── databricks.yml
├── setup.py
├── pyproject.toml
├── README.md
└── .gitignore
```

## Project requirements: 
use-case:
- Data about diabetes patients coming from PostgreSQL
- Ingest using LakeFlow (Bronze)
- Transform with PySpark to Silver
- Clean, validate, standardize medical data
- Produce a Gold table for analytics

This answer will include:
- every folder
- every file
- with real content
- production-style Databricks examples
- but not overwhelming — clean + practical

## COMPLETE PROJECT (EVERY FILE FILLED OUT)
### lakeflow related pipelines
#### lakeflow/ (Ingestion config)
I am going to use AWS RDS postgre server for this 
```bash
connections:
  diabetes_pg:
    type: postgresql
    catalog: main
    schema: bronze_diabetes
    options:
      host: "YOUR_PG_HOST"
      port: "5432"
      database: "healthdb"
    credentials:
      username: "{{secrets/pg/username}}"
      password: "{{secrets/pg/password}}"
```
#### lakeflow/ingest/patient_ingest.yaml
This pipeline ingests the raw table from Postgres into Bronze.
```bash
pipeline:
  name: diabetes_patient_ingest
  schedule: daily
  target: main.bronze_diabetes.patients_raw

  read:
    connection: diabetes_pg
    table: public.diabetes_patients

  write:
    format: delta
    mergeKeys:
      - patient_id
```
#### lakeflow/pipelines/README.md

### src/your_app/ (Your PySpark application code)
#### src/your_app/__init__.py : empty file
#### src/your_app/bronze_to_silver/process_patients.py
Bronze → Silver transformation
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, to_date, when
)

def process():
    spark = SparkSession.builder.getOrCreate()

    # Read Bronze table produced by LakeFlow ingestion
    df = spark.read.table("main.bronze_diabetes.patients_raw")

    # CLEANING & STANDARDIZATION
    df_clean = (
        df
        .withColumn("patient_id", trim(col("patient_id")))
        .withColumn("name", trim(col("name")))
        .withColumn("gender",
            lower(trim(col("gender")))
        )
        .withColumn("diagnosis_date",
            to_date(col("diagnosis_date"), "yyyy-MM-dd")
        )
        .withColumn("blood_sugar",
            col("blood_sugar").cast("double")
        )
        # Standardizing gender
        .withColumn(
            "gender",
            when(col("gender").isin("male", "m"), "male")
            .when(col("gender").isin("female", "f"), "female")
            .otherwise("unknown")
        )
        # Handle null or corrupted blood sugar
        .withColumn(
            "blood_sugar",
            when(col("blood_sugar") < 10, None)  # unrealistic values
            .otherwise(col("blood_sugar"))
        )
    )

    # Write to Silver
    df_clean.write.mode("overwrite").format("delta").saveAsTable(
        "main.silver_diabetes.patients_clean"
    )

    print("✅ Silver table created: main.silver_diabetes.patients_clean")
```
#### src/your_app/bronze_to_silver/diabetes_metrics.py
Analytics (Silver → Gold)
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

def build_metrics():
    spark = SparkSession.builder.getOrCreate()

    df = spark.read.table("main.silver_diabetes.patients_clean")

    metrics = df.groupBy("gender").agg(
        avg(col("blood_sugar")).alias("avg_blood_sugar")
    )

    metrics.write.mode("overwrite").saveAsTable(
        "main.gold_diabetes.patient_metrics"
    )

    print("✅ Gold table created: main.gold_diabetes.patient_metrics")
```
### Utiltities
#### src/your_app/utils/spark_init.py
```python
from pyspark.sql import SparkSession

def get_spark():
    return SparkSession.builder.getOrCreate()
```
#### src/your_app/config/settings.py
```python
BRONZE_TABLE = "main.bronze_diabetes.patients_raw"
SILVER_TABLE = "main.silver_diabetes.patients_clean"
GOLD_TABLE = "main.gold_diabetes.patient_metrics"
```
### notebooks/
You can keep these empty or add testing notebooks.
### tests/
#### tests/test_process_patients.py
```python
def test_dummy():
    assert 1 == 1
```
### jobs/
#### jobs/process_patients_job.yaml
```bash
jobs:
  diabetes_etl:
    tasks:
      - task_key: bronze_to_silver
        python_wheel_task:
          package_name: your_app
          entry_point: your_app.bronze_to_silver.process_patients:process
        compute:
          serverless: true

      - task_key: build_metrics
        depends_on:
          - task_key: bronze_to_silver
        python_wheel_task:
          package_name: your_app
          entry_point: your_app.bronze_to_silver.diabetes_metrics:build_metrics
        compute:
          serverless: true
```
### Packaging & Deployment
#### setup.py
```python
from setuptools import setup, find_packages

setup(
    name="your_app",
    version="0.1.0",
    packages=find_packages("src"),
    package_dir={"": "src"},
)

```
#### pyproject.toml
```bash
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"
```
#### databricks.yml
```bash

bundle:
  name: diabetes_project

artifacts:
  diabetes_app:
    type: python_wheel
    build: "python setup.py bdist_wheel"

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: YOUR_DATABRICKS_URL

resources:
  jobs:
    diabetes_etl:
      <<: *dev
```
#### .gitignore
```bash
__pycache__/
*.pyc
*.pyo
dist/
build/
.DS_Store
```
#### Final Summary (Full E2E Flow)
Here’s the whole pipeline, human-style:
PostgreSQL ──> LakeFlow Connection
                │
                ▼
        LakeFlow Ingest Pipeline
                │
                ▼
       Bronze Table (patients_raw)
                │
                ▼
     PySpark transform (clean data)
                │
                ▼
    Silver Table (patients_clean)
                │
                ▼
     PySpark metrics generation
                │
                ▼
      Gold Table (patient_metrics)
- You now have a full production-ready Databricks + LakeFlow + PySpark project
- With clean folder structure
- Every file filled out
- Bronze → Silver → Gold ETL logic included
- Realistic health/diabetes dataset transformations


# “How do I test my PySpark application locally during development?”
# “How do I know if it’s working correctly before I push anything to Databricks?”
## Step 1 — Install the right local environment
Inside your project folder:
```bash
pip install pyspark==3.5.0
pip install delta-spark
pip install pytest
```
NOTE : You need to match your pyspark, scala and java version with databricks in your local machine so that when you test your spark application the outputs are not only predictable but also consistent.
## Step 2 — Modify your Spark initialization for LOCAL mode
In your spark_init.py file:
```python
from pyspark.sql import SparkSession

def get_spark():
    return (
        SparkSession.builder
            .appName("local-dev")
            .master("local[*]")  # ✅ local Spark
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
            .getOrCreate()
    )
```
This makes your app run locally exactly like Databricks Delta runtime.
## Step 3 — Create a local dev input folder (acts like your Bronze table)
Example:
```bash
dev_data/
  patients_raw.parquet
```
To generate fake data:
```python
df.write.mode("overwrite").parquet("dev_data/patients_raw.parquet")
```

Or just grab your Postgres data and export it once.

## Step 4 — Modify your transformation script so you can pass local paths
Example: process_patients.py
```python
def process(input_path=None, output_path=None):
    spark = get_spark()

    if input_path:
        df = spark.read.parquet(input_path)
    else:
        df = spark.read.table("main.bronze_diabetes.patients_raw")  # DBricks

    df_clean = ... # same logic

    if output_path:
        df_clean.write.mode("overwrite").parquet(output_path)
    else:
        df_clean.write.mode("overwrite").saveAsTable("main.silver_diabetes.patients_clean")

```
### Local test run
```bash
python src/your_app/bronze_to_silver/process_patients.py \
    --input dev_data/patients_raw.parquet \
    --output dev_data/patients_clean.parquet
```
Boom — your Silver output appears locally.
## Step 5 — Use pytest for repeatable unit tests
Inside tests/test_process_patients.py
```python
from your_app.bronze_to_silver.process_patients import process
from your_app.utils.spark_init import get_spark
import os

def test_process_patients_local(tmp_path):
    spark = get_spark()

    # make fake input
    df = spark.createDataFrame([
        ("P001", " John Doe ", 45, "M", "2023-01-01", 150.0, 6.8),
        ("P002", "Jane Doe", 50, "Female", "2023-02-10", 220.0, 8.1),
    ], ["patient_id", "name", "age", "gender", "diagnosis_date", "blood_sugar", "hba1c"])

    input_path = str(tmp_path / "input.parquet")
    output_path = str(tmp_path / "output.parquet")

    df.write.parquet(input_path)

    # run transformation
    process(input_path=input_path, output_path=output_path)

    # validate results
    out = spark.read.parquet(output_path)
    assert out.count() == 2
```
Now just run:
```bash
pytest -q
```
## Step 6 — For a complete local simulation, create a small local “Lakehouse”
```bash
local_lakehouse/
  bronze/
  silver/
  gold/
```
Then in your script:
```python
process(
    input_path="local_lakehouse/bronze/patients_raw",
    output_path="local_lakehouse/silver/patients_clean"
)
```
## Step 7 — When your local tests pass → deploy to Databricks
Only after:
- Script runs locally
- Data outputs look correct
- pytest passes
THEN you:
```bash
databricks bundle deploy
databricks jobs run-now ...
```

This gives total confidence your ETL works.

## Final guide you should follow:
- Develop locally (fast)
- Write PySpark unit tests (reliable)
- Run with local Spark (cheap)
- Deploy to Databricks only when stable (safe)
This is how 99% of real Databricks engineers work.

# Is bundling my pyspark application a good idea or should I just use notebooks provided by databricks that can easily be created from the databricks workspaces?
### If you’re just learning / experimenting / doing small ETL
Use Databricks notebooks.
Why?
- Faster feedback
- Auto Spark session
- Auto widgets
- Easy DBFS browsing
- Magic commands %sql, %pip
- No packaging overhead
- Easy to debug visually
- You can schedule notebooks as jobs directly
For most junior-to-mid engineers building simple ETL pipelines, notebooks are more than enough.
If your boss isn’t asking for CI/CD, unit tests, or versioned releases, use notebooks.
### If you’re building a real production system
Bundle your PySpark app.
Why?
- Reusable across multiple teams/jobs
- Versioned & deployable like a real software artifact
- Easily testable locally (pytest, local Spark)
- Fits into CI/CD pipelines
- Easier for large engineering teams to collaborate
- No notebook merge conflict hell
- Gives you clean dev-test-prod separation
- Works with LakeFlow, Jobs 2.0, Repos, bundles, wheels
- Best for long-term maintainability
Once your project grows beyond a few notebooks, notebooks start to feel like duct tape.
Production teams bundle their code into:
- Python wheels
- Databricks bundles
- Unity Catalog volumes / libraries
That’s standard engineering.



# Will these pipelines run automatically or do I need setup?
Short answer: They do NOT run magically by themselves.
You need to set up Databricks Bundles + GitHub repo + Deployment pipeline.
Out of the box:
- The .yaml files you wrote (LakeFlow connections, pipelines, jobs)
don’t auto-deploy.
- The databricks.yml also doesn't deploy automatically until you wire it to CI/CD.
- The wheel packaging (setup.py) only runs when triggered.
- The jobs only run when you deploy the bundle or schedule it.
## How to set up your GitHub repo for dev → QA → prod
The cleanest structure used in real data engineering teams is:
```bash
main (PROD)
├── qa
└── dev
```
### Branch Purpose
| Branch | Purpose                | Who deploys                             |
| ------ | ---------------------- | --------------------------------------- |
| `dev`  | Your ongoing work      | Auto deploy to Databricks-dev workspace |
| `qa`   | Testing, UAT, pre-prod | Auto deploy to Databricks-qa workspace  |
| `main` | Production             | Auto deploy to Prod workspace           |
### Databricks Workspace separation
#### 3 separate workspaces
- databricks-dev
- databricks-qa
- databricks-prod
### Databricks Bundles needs environment sections
Your databricks.yml needs to look like this:
```bash
bundle:
  name: diabetes_project

artifacts:
  diabetes_app:
    type: python_wheel
    build: "python setup.py bdist_wheel"

targets:
  dev:
    mode: development
    workspace:
      host: https://dev.cloud.databricks.com
    default: true

  qa:
    mode: development
    workspace:
      host: https://qa.cloud.databricks.com

  prod:
    mode: production
    workspace:
      host: https://prod.cloud.databricks.com
```
Now Databricks automatically knows:
```bash
databricks bundle deploy -t dev
databricks bundle deploy -t qa
databricks bundle deploy -t prod
```
### GitHub Repo Folder Structure
Your repo root stays exactly as you designed earlier:
```bash
/lakeflow
/src
/tests
/jobs
notebooks
databricks.yml
pyproject.toml
setup.py
```
But the magic happens via GitHub Actions.
### GitHub Actions setup (THE MAGIC)
Inside your repo create:
```bash
.github/workflows/databricks-deploy.yml
```
Paste this (real, production-ready)
```bash
name: Deploy Databricks Bundle

on:
  push:
    branches:
      - dev
      - qa
      - main

jobs:
  deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repo
        uses: actions/checkout@v3

      - name: Install databricks CLI
        run: pip install databricks-cli databricks-bundles

      - name: Set environment target
        id: envselect
        run: |
          if [[ "${GITHUB_REF##*/}" == "dev" ]]; then
            echo "target=dev" >> $GITHUB_OUTPUT
          elif [[ "${GITHUB_REF##*/}" == "qa" ]]; then
            echo "target=qa" >> $GITHUB_OUTPUT
          else
            echo "target=prod" >> $GITHUB_OUTPUT
          fi

      - name: Deploy bundle
        run: |
          databricks bundle deploy -t ${{ steps.envselect.outputs.target }}
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}

```
#### What this GitHub workflow does
- push to dev → deploy to dev workspace
- push to qa → deploy to qa workspace
- merge to main → deploy to prod workspace
No manual CLI, no local commands.
CI/CD handles everything.

## full GitHub repo template
### Github repo structure : 
```bash
your-databricks-diabetes-project/
│
├── lakeflow/
│   ├── connections/
│   │   └── postgres_conn.yaml
│   ├── ingest/
│   │   └── patient_ingest.yaml
│   └── pipelines/
│       └── README.md
│
├── src/
│   └── your_app/
│       ├── __init__.py
│       ├── bronze_to_silver/
│       │   ├── __init__.py
│       │   ├── process_patients.py
│       │   └── diabetes_metrics.py
│       ├── utils/
│       │   ├── __init__.py
│       │   └── spark_init.py
│       └── config/
│           └── settings.py
│
├── jobs/
│   └── process_patients_job.yaml
│
├── tests/
│   ├── test_process_patients.py
│   └── test_utils.py
│
├── notebooks/
│   ├── exploration/
│   └── debugging/
│
├── databricks.yml
├── pyproject.toml
├── setup.py
├── .gitignore
└── .github/
    └── workflows/
        └── databricks-deploy.yml
```
### Complete Databricks Bundle With Environments
#### databricks.yml
```bash
bundle:
  name: diabetes_project

artifacts:
  diabetes_app:
    type: python_wheel
    build: "python setup.py bdist_wheel"

targets:
  dev:
    mode: development
    workspace:
      host: https://<your-dev-workspace>.cloud.databricks.com
    default: true
    variables:
      catalog_prefix: "dev"

  qa:
    mode: development
    workspace:
      host: https://<your-qa-workspace>.cloud.databricks.com
    variables:
      catalog_prefix: "qa"

  prod:
    mode: production
    workspace:
      host: https://<your-prod-workspace>.cloud.databricks.com
    variables:
      catalog_prefix: "prod"

resources:
  jobs:
    diabetes_etl:
      name: "diabetes_etl_job"
      tasks:
        - task_key: bronze_to_silver
          python_wheel_task:
            package_name: your_app
            entry_point: your_app.bronze_to_silver.process_patients:process
          compute:
            serverless: true

        - task_key: build_metrics
          depends_on:
            - bronze_to_silver
          python_wheel_task:
            package_name: your_app
            entry_point: your_app.bronze_to_silver.diabetes_metrics:build_metrics
          compute:
            serverless: true
```
The variable catalog_prefix changes based on environment
- Dev writes to dev.bronze_diabetes…
- QA writes to qa.silver_diabetes…
- Prod writes to prod.gold_diabetes…
#### LakeFlow Connection (ready for multi-env) : lakeflow/connections/postgres_conn.yaml
```bash
connections:
  diabetes_pg:
    type: postgresql
    catalog: "{{bundle.target.variables.catalog_prefix}}"
    schema: bronze_diabetes
    options:
      host: "YOUR_PG_HOST"
      port: "5432"
      database: "healthdb"
    credentials:
      username: "{{secrets/pg/username}}"
      password: "{{secrets/pg/password}}"
```
#### LakeFlow Ingest Pipeline : lakeflow/ingest/patient_ingest.yaml
```bash
pipeline:
  name: diabetes_patient_ingest
  schedule: daily

  target: "{{bundle.target.variables.catalog_prefix}}.bronze_diabetes.patients_raw"

  read:
    connection: diabetes_pg
    table: public.diabetes_patients

  write:
    format: delta
    mergeKeys:
      - patient_id
```
#### Production-grade GitHub Actions : .github/workflows/databricks-deploy.yml
```bash
name: Deploy Databricks Bundle

on:
  push:
    branches:
      - dev
      - qa
      - main

jobs:
  deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout
        uses: actions/checkout@v3

      - name: Install Databricks CLI
        run: |
          pip install databricks cli databricks-bundles setuptools wheel

      - name: Select environment
        id: env
        run: |
          branch="${GITHUB_REF##*/}"
          if [[ "$branch" == "dev" ]]; then
            echo "target=dev" >> $GITHUB_OUTPUT
          elif [[ "$branch" == "qa" ]]; then
            echo "target=qa" >> $GITHUB_OUTPUT
          else
            echo "target=prod" >> $GITHUB_OUTPUT
          fi

      - name: Deploy Bundle
        run: |
          databricks bundle deploy -t ${{ steps.env.outputs.target }}
          databricks bundle run -t ${{ steps.env.outputs.target }}
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
```
- Automatic deployments from branch → environment
- Single source of truth CI/CD pipeline
- Automatically runs job after deploy
#### GitHub Branch Strategy
default setup :
```bash
dev → qa → main
```
Branch protection rules :
Enable for:
- main
- qa
Rules:
- Require PR before merging
- Require CI to pass
- Require reviews
- Block direct pushes
#### Secrets You MUST Add in GitHub
In GitHub → Settings → Secrets → Actions:
| Secret             | Why                                                                                  |
| ------------------ | ------------------------------------------------------------------------------------ |
| `DATABRICKS_HOST`  | URL of workspace (dev/qa/prod share same var because env is chosen by bundle target) |
| `DATABRICKS_TOKEN` | PAT token to deploy                                                                  |
| `PG_USERNAME`      | PostgreSQL                                                                           |
| `PG_PASSWORD`      | PostgreSQL                                                                           |
#### Databricks Setup You Must Do
In User Settings → Developer:
- Create a PAT token
- Add a git folder for repos
- Create Unity Catalog catalogs:
    - dev
    - qa
    - prod
- Create schemas:
    - bronze_diabetes
    - silver_diabetes
    - gold_diabetes
- Add secrets:
```bash
secrets/pg/username
secrets/pg/password
```
#### Full Deployment Flow — Visual
```bash
              ┌────────────┐
              │   DEV BRANCH│
              └──────┬─────┘
                     │ push
                     ▼
           GitHub Actions (dev)
                     │
                     ▼
     Deploy bundle to databricks-dev workspace
                     │
                     ▼
         Run ingestion + ETL jobs (dev)

(Once stable)
        PR → qa branch
                     │ push
                     ▼
           GitHub Actions (qa)
                     │
                     ▼
     Deploy bundle to databricks-qa workspace
                     │
                     ▼
           Run pipelines (QA testing)

(Once approved)
        PR → main
                     │ push
                     ▼
       GitHub Actions (prod)
                     │
                     ▼
     Deploy bundle to databricks-prod workspace
                     │
                     ▼
        Production LakeFlow + ETL runs

```

## How to can I run test cases that I wrote automatically using ci/cd in qa branch after deployement of the pyspark application on databricks qa workspace?
### Goal : 
- The CI/CD pipeline deploys your bundle to Databricks QA
- THEN runs your PySpark tests in Databricks, not GitHub
- Tests use a cluster/serverless compute on QA
- CI fails if tests fail
### Run tests as a Databricks Workflow job (BEST for QA)
- Runs on real Databricks compute
- Sees Unity Catalog tables
- Mirrors production
- CI can wait for job to finish
#### STEP 1 — Create a Databricks Test Job in : databricks.yml
```bash
resources:
  jobs:
    qa_tests:
      name: "diabetes_qa_tests"
      tasks:
        - task_key: run_pytests
          python_wheel_task:
            package_name: your_app
            entry_point: your_app.tests.run_tests:main
          compute:
            serverless: true
```
#### STEP 2 — Add a test runner Python file in your wheels : create src/your_app/tests/run_tests.py
```python
import pytest
import sys

def main():
    # Run all tests in the wheel’s tests folder
    exit_code = pytest.main(["-q"])
    if exit_code != 0:
        sys.exit(exit_code)

```
- This lets Databricks run your tests as a workflow task
- SparkSession will be available
- Unity Catalog will be available
- Delta tables will be available
#### STEP 3 — Update your GitHub CI/CD to run tests AFTER deployment
Modify .github/workflows/databricks-deploy.yml:
```bash
- name: Run QA Tests (only on qa branch)
  if: github.ref == 'refs/heads/qa'
  run: |
    databricks jobs run-now --job-id $(databricks jobs list --output JSON | jq '.jobs[] | select(.settings.name=="diabetes_qa_tests").job_id')
  env:
    DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
    DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
```
- This triggers your Databricks test job
- CI waits for test results
- A failed job = failed CI status
#### STEP 4 — Your tests should NOT use local files
PySpark tests must use:
```python
spark = SparkSession.builder.getOrCreate()
```
or use fixture:
```python
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()
```
#### STEP 5 — Example QA Test Running in Databricks
tests/test_process_patients.py
```python
def test_blood_sugar_cleaning(spark):
    df = spark.createDataFrame([
        ("p1", "John", "M", "2024-01-10", 5.0),
        ("p2", "Jane", "F", "2024-01-11", 180.0)
    ], ["patient_id","name","gender","diagnosis_date","blood_sugar"])

    from your_app.bronze_to_silver.process_patients import clean

    result = clean(df)

    assert result.filter("blood_sugar is null").count() == 1
```
- This test will run on Databricks QA
- Test can read from QA catalog
- Failures stop the CI pipeline
#### Final QA Workflow (This is what your team will proudly present in interviews)
**Push to qa branch**
- GitHub deploys bundle to Databricks QA
- GitHub triggers QA Test Job
- Databricks spins up serverless cluster
- Runs pytest in Databricks
- Reports pass/fail to GitHub
- Blocks merge to main unless tests pass
This is exactly how production data engineering teams do it at fintech, healthcare, SaaS, and enterprise companies today.